In [1]:
import torch
from tqdm import trange
import numpy as np
import scipy
import sys
torch.set_grad_enabled(False)
import torch.nn as nn


In [2]:
N = 1024
B = 16
H = 768
N1 = int(np.sqrt(N))

u = (torch.randn((B, H, N), dtype=torch.cfloat, device='cpu')).real
k = (torch.ones((H, N), dtype=torch.cfloat, device='cpu')).real

In [3]:
def fft_matrix(N):
    n = torch.arange(N)
    k = n.view(-1, 1)
    M = torch.exp(-2j * torch.pi * n * k / N).real
    return M

def compute_twiddle_factors_fft(n, m):
    """Compute the twiddle factors of size n x m"""
    n_a = torch.arange(n).view(-1, 1)
    m_a = torch.arange(m)
    N = n * m
    M = torch.exp(-2j * torch.pi * n_a * m_a / N).real
    return M

def ifft_matrix(N):
    n = torch.arange(N)
    k = n.view(-1, 1)
    M = torch.exp(2j * torch.pi * n * k / N).real
    return M

def compute_twiddle_factors_ifft(n, m):
    """Compute the twiddle factors of size n x m"""
    n_a = torch.arange(n).view(-1, 1)
    m_a = torch.arange(m)
    N = n * m
    M = torch.exp(2j * torch.pi * n_a * m_a / N).real
    return M / N

In [4]:
def pytorch_test(u, k):
    ############# GET THE INPUTS #############
    f_mat = fft_matrix(N1)
    finv_mat = ifft_matrix(N1)
    
    # Normalization factor to make IFFT exact inverse of FFT
    twiddle_factors_fft = compute_twiddle_factors_fft(N1, N1).to(u.device)
    twiddle_factors_ifft = compute_twiddle_factors_ifft(N1, N1).to(u.device)

    k_f = torch.fft.fft(k, n = N).real
    k_fT = k_f.reshape(H, N1, N1).transpose(-1, -2).real
    print(f"DTYPES: {u.dtype=}, {k.dtype=}, {f_mat.dtype=}, {finv_mat.dtype=}, {twiddle_factors_fft.dtype=}, {twiddle_factors_ifft.dtype=}, {k_f.dtype=}, {k_fT.dtype=}")

    ############# COMPUTE OUTPUT #############
    # step 1. FFT(U) using FFT matrix
    # compute the FFT
    x = u.reshape(B, H, N1, N1)
    x = x.transpose(-1, -2)
    x = torch.einsum('...i,ij->...j', x, f_mat)
    x = x.transpose(-1, -2)
    x = x * twiddle_factors_fft # (H, sqrt_N, sqrt_N) * (sqrt_N, sqrt_N), pointwise
    x = torch.einsum('...i,ij->...j', x, f_mat)
    # x = x.transpose(-1, -2)

    # pointwise multiplication 
    k_f = k_f.reshape(H, N1, N1) # to match the shape of x
    x = x * k_f

    # compute the IFFT
    # x = x.transpose(-1, -2)
    x = x @ finv_mat
    x = x.transpose(-1, -2)
    x = x * twiddle_factors_ifft # (H, sqrt_N, sqrt_N) * (sqrt_N, sqrt_N), pointwise
    x = x @ finv_mat
    x = x.transpose(-1, -2) # necessary to complete the ifft

    x = x.reshape(B, H, N)

    # return torch.fft.rfft(u, n = N)
    return x


In [5]:
def ref_fftconv_test(u, k, N):
    L = u.shape[-1]
    u_f = torch.fft.fft(u, n = N)
    k_f = torch.fft.fft(k, n = N)
    y_f = u_f * k_f
    y = torch.fft.ifft(y_f, n = N).real[..., :L].to(u.dtype).contiguous()
    return y

# apply conv on each dimension
def torch_conv(u, k):
    conv = nn.Conv1d(
        in_channels=H,
        out_channels=H,
        kernel_size=N,
        groups=H,
        padding=0,
        bias=False,
    )
    conv.weight.data = k.unsqueeze(1).flip(-1)
    L = u.shape[-1]
    y = conv(u)[..., :L]
    return y

V_real = pytorch_test(u, k)
V_test = ref_fftconv_test(u, k, N).real 
V_conv = torch_conv(u, k)
print(torch.allclose(V_real, V_test, atol=1e-3))
print(V_real[0, 0:6, :4])
print(V_conv[0, 0:6, :4])
print(V_test[0, 0:6, :4])

DTYPES: u.dtype=torch.float32, k.dtype=torch.float32, f_mat.dtype=torch.float32, finv_mat.dtype=torch.float32, twiddle_factors_fft.dtype=torch.float32, twiddle_factors_ifft.dtype=torch.float32, k_f.dtype=torch.float32, k_fT.dtype=torch.float32
True
tensor([[ -9.3377,  -9.3377,  -9.3377,  -9.3377],
        [-15.0772, -15.0772, -15.0772, -15.0772],
        [ 12.1740,  12.1740,  12.1740,  12.1740],
        [-23.0644, -23.0644, -23.0644, -23.0644],
        [  7.0969,   7.0969,   7.0969,   7.0969],
        [ 49.0194,  49.0194,  49.0194,  49.0194]])
tensor([[ -9.3377],
        [-15.0772],
        [ 12.1740],
        [-23.0644],
        [  7.0969],
        [ 49.0194]])
tensor([[ -9.3377,  -9.3377,  -9.3377,  -9.3377],
        [-15.0772, -15.0772, -15.0772, -15.0772],
        [ 12.1740,  12.1740,  12.1740,  12.1740],
        [-23.0644, -23.0644, -23.0644, -23.0644],
        [  7.0969,   7.0969,   7.0969,   7.0969],
        [ 49.0194,  49.0194,  49.0194,  49.0194]])
